# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

**TODO:** Replace this text with an evidence map: one bullet for each of Lectures 01 to 06. For 01, 02, 04, 05, and 06, name one concrete technique or file and say in one sentence where this notebook or its outputs use it. For 03, complete the NumPy calculation after Task 4.3 and explain your axis choice in one sentence here. The lectures cover the command line and Python basics (01); Git, functions, and files (02); environments, shell pipelines, and NumPy (03); Jupyter and labeled pandas data (04); data contracts, cleaning, and validation (05); and joins, reshaping, and concatenation (06). Do not paste earlier solutions.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [ ]:
def profile(name, table):
    return {
        "table": name, 
        "rows": table.shape[0],
        "columns": table.shape[1],
        "exact_duplicate_rows": int(table.duplicated().sum()),
    }

results = []

for name in manifest["files"]:
    results.append(profile(name, raw[Path(name).stem]))

table_profile = pd.DataFrame(results)
table_profile.to_csv(PROFILE_PATH, index=False)   
table_profile
    

,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

**Grain/key**
- a1c_wide: grain - one row per patient; key - patient_id (single). Then reshaped to long - grain - one row per patient  - visit; key - patient_id and visit (composite).
- patients: grain - one row per patient; key - patient_id (single).
- sites: grain - one row per site; key - site_code (single).
- visits_north: grain - one row per patient visit; key - patient_id and visit (composite). stacked on south.
- visits_south grain - one row per patient visit; key - patient_id and visit (composite). stacked on north.

**Violations**
- Exact repeats: Completely identical rows, 45 in patients, 35 per visit row.
- Rechecks: Same visit but multiple BP readings, so patient_id and visit are duplicated
- Site versions: some sites were renamed and the data has historical names

**Cardinality**
- visits to patients: m:1
- patients to sites: m:1  
- visits to the long-form A1c results: 1:1

### 2.2 Stack the two visit files

In [ ]:
visits_north = raw["visits_north"].copy(deep=True)
visits_south = raw["visits_south"].copy(deep=True).rename(columns={"wt_kg": "weight_kg"})

visits_north["source_file"] = "visits_north.csv"
visits_south["source_file"] = "visits_south.csv"

visits_raw = pd.concat([visits_north, visits_south], ignore_index=True)
visits_raw

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
...,...,...,...,...,...,...,...
57366,P24976,week12,2026-05-14,10:50,153,78.3,visits_south.csv
57367,P19276,baseline,2026-01-22,15:35,160,122.2,visits_south.csv
57368,P21334,week12,2026-05-05,09:40,129,98.2,visits_south.csv
57369,P03524,baseline,2026-02-06,10:05,160,117.6,visits_south.csv


### 2.3 Audit the raw tables

In [ ]:
pat = raw["patients"]
sit = raw["sites"]
vis = visits_raw
sbp = pd.to_numeric(vis["sbp_mmhg"], errors="coerce")
visit_dict = {"baseline":"baseline", "week12":"week12", "bl":"baseline", "wk12":"week12", "week 12":"week12"}
tmp = vis.assign(visit=vis["visit"].str.strip().str.lower().map(visit_dict))
rows = [
    ("patients: gender values that are not a code", ((pat["gender"].str.strip() != "") & (~pat["gender"].isin(["M", "F", "X"]))).sum()),
    ("patients: empty gender", (pat["gender"].str.strip() == "").sum()),
    ("patients: birth_year sentinel 9999", (pat["birth_year"] == "9999").sum()),
    ("patients: site codes needing normalization", (pat["site_code"].str.upper().str.replace(" ","").str.replace("-", "") != pat["site_code"]).sum()),
    ("sites: rows with a repeated site_code", sit.duplicated(subset=["site_code"], keep=False).sum()),
    ("visits: visit labels that are not baseline or week12", (~vis["visit"].isin(["baseline", "week12"])).sum()),
    ("visits: sbp sentinel 999", (vis["sbp_mmhg"] == "999").sum()),
    ("visits: sbp outside 70 through 250", ((sbp != 999) & ((sbp < 70) | (sbp > 250))).sum()),
    ("visits: date parse failures",(~((vis["visit_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")) & (pd.to_datetime(vis["visit_date"], format="%Y-%m-%d", errors="coerce").notna()))).sum()),
    ("visits: rows in repeated patient_id and visit pairs",tmp.duplicated(subset=["patient_id", "visit"], keep=False).sum())
]

issue_audit = pd.DataFrame(rows, columns=["issue", "count"])
issue_audit.to_csv(AUDIT_PATH, index=False)
assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."
issue_audit

,issue,count
0,patients: gender values that are not a code,6968
1,patients: empty gender,486
2,patients: birth_year sentinel 9999,371
3,patients: site codes needing normalization,1811
4,sites: rows with a repeated site_code,4
5,visits: visit labels that are not baseline or ...,2309
6,visits: sbp sentinel 999,341
7,visits: sbp outside 70 through 250,116
8,visits: date parse failures,169
9,visits: rows in repeated patient_id and visit ...,380


## Task 3: Clean

### 3.1 Clean the patients

In [ ]:
patients_clean = raw["patients"].copy(deep=True)
patients_clean = patients_clean.drop_duplicates(keep="first")
gender_series = patients_clean["gender"].str.strip().str.lower()
gender_map = {"nb": "X", "f": "F", "female": "F", "woman": "F", "w": "F", "m": "M", "male": "M", "man": "M", "x": "X", "nonbinary": "X", "non-binary": "X"}
patients_clean["gender"] = gender_series.map(gender_map)
patients_clean["gender"] = patients_clean["gender"].astype("category")
patients_clean["birth_year"] = pd.to_numeric(patients_clean["birth_year"], errors="coerce")
patients_clean.loc[patients_clean["birth_year"] == 9999, "birth_year"] = np.nan
patients_clean["birth_year"] = patients_clean["birth_year"].astype("Int64")
patients_clean["site_code"] = (patients_clean["site_code"].str.upper().str.replace("-", "").str.replace(" ", ""))
patients_clean = patients_clean.reset_index(drop=True)
patients_metrics = [
    ("patients", len(patients_clean)),
    ("birth_year missing", patients_clean["birth_year"].isna().sum()),
    ("birth_year min", patients_clean["birth_year"].min()),
    ("birth_year max", patients_clean["birth_year"].max()),
    ("site_code values", patients_clean["site_code"].nunique()),
    ("patients at N01", (patients_clean["site_code"] == "N01").sum()),
    ("patients at S02", (patients_clean["site_code"] == "S02").sum()),
    ("patients at E04", (patients_clean["site_code"] == "E04").sum())
]

In [ ]:
patients_summary = pd.DataFrame(patients_metrics, columns=["metric", "value"])
patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)
gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)
display(patients_clean["gender"].cat.categories)   # expect ['F', 'M', 'X']
display(patients_summary)
gender_counts

Index(['F', 'M', 'X'], dtype='str')

,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


,gender,count,code
0,F,14413,0
1,M,13955,1
2,NaN,863,-1
3,X,769,2


### 3.2 Clean the visits

In [ ]:
v_df = visits_raw.copy(deep=True)
v_df = v_df.drop_duplicates(keep="first")
visit_stripped = v_df["visit"].str.strip().str.lower()
v_df["visit"] = visit_stripped.map(visit_dict)
v_df["sbp_mmhg"] = pd.to_numeric(v_df["sbp_mmhg"], errors="coerce")
invalid_sbp = (v_df["sbp_mmhg"] < 70) | (v_df["sbp_mmhg"] > 250) | (v_df["sbp_mmhg"] == 999)
v_df.loc[invalid_sbp, "sbp_mmhg"] = np.nan
v_df["sbp_mmhg"] = v_df["sbp_mmhg"].astype("Int64")
v_df["weight_kg"] = pd.to_numeric(v_df["weight_kg"], errors="coerce")
is_exact_format = v_df["visit_date"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}") == True
parsed_dates = pd.to_datetime(v_df["visit_date"], format="%Y-%m-%d", errors="coerce")
v_df["visit_date"] = parsed_dates.where(is_exact_format)

In [ ]:
rechecks_mask = v_df.duplicated(subset=["patient_id", "visit"], keep=False)
print("Clinical recheck rows to be resolved:\n", v_df[rechecks_mask])
v_df = v_df.sort_values(by="visit_time")
v_df = v_df.drop_duplicates(subset=["patient_id", "visit"], keep="last")

Clinical recheck rows to be resolved:
       patient_id     visit visit_date visit_time  sbp_mmhg  weight_kg  \
4         P00004  baseline 2026-02-25      10:55       145      106.2   
5         P00004  baseline 2026-02-25      11:15       142      106.2   
84        P00088  baseline 2026-03-10      09:30       153       68.2   
85        P00088  baseline 2026-03-10      09:50       152       68.2   
1208      P01341    week12 2026-05-26      15:10       121       72.3   
...          ...       ...        ...        ...       ...        ...   
55282     P28051    week12 2026-04-16      09:35       126       92.7   
55472     P28232  baseline 2026-01-12      10:00       153       61.0   
55473     P28232  baseline 2026-01-12      10:10       159       61.0   
56248     P28953  baseline 2026-01-21      10:45       151       74.8   
56249     P28953  baseline 2026-01-21      10:30       153       74.8   

            source_file  
4      visits_north.csv  
5      visits_north.csv  
84    

In [ ]:
v_df = v_df.sort_values(by=["patient_id", "visit"])
columns_order = ["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]
visits_clean = v_df[columns_order].reset_index(drop=True)

visits_metrics = [
    ("visits", len(visits_clean)),
    ("baseline visits", (visits_clean["visit"] == "baseline").sum()),
    ("week12 visits", (visits_clean["visit"] == "week12").sum()),
    ("visit_date missing", visits_clean["visit_date"].isna().sum()),
    ("visit_date min", visits_clean["visit_date"].min()),
    ("visit_date max", visits_clean["visit_date"].max()),
    ("sbp_mmhg missing", visits_clean["sbp_mmhg"].isna().sum()),
    ("sbp_mmhg min", visits_clean["sbp_mmhg"].min()),
    ("sbp_mmhg max", visits_clean["sbp_mmhg"].max()),
    ("weight_kg missing", visits_clean["weight_kg"].isna().sum()),
    ("weight_kg mean", round(visits_clean["weight_kg"].mean(), 2)),
    ("visits from visits_north.csv", (visits_clean["source_file"] == "visits_north.csv").sum()),
    ("visits from visits_south.csv", (visits_clean["source_file"] == "visits_south.csv").sum())
]

visits_summary = pd.DataFrame(visits_metrics, columns=["metric", "value"])
visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)
visits_summary

,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [ ]:
decisions_data = [
    {
        "table": "patients",
        "issue": "exact duplicate rows",
        "decision": "Removed exact duplicate rows using drop_duplicates, keeping the first occurrence.",
        "reason": "Identical rows are not possible."
    },
    {
        "table": "patients",
        "issue": "gender spellings that are not a code",
        "decision": "Stripped, lowercased, mapped known non-standard text to unified codes, and set unmatched entries to missing.",
        "reason": "Correcting categorial gender information."
    },
    {
        "table": "patients",
        "issue": "site codes needing normalization",
        "decision": "Converted strings to uppercase and stripped out all stray hyphens and spaces.",
        "reason": "Text fields typed by hand require rigorous format cleaning to support precise joining operations later."
    },
    {
        "table": "patients",
        "issue": "birth_year sentinel 9999",
        "decision": "Coerced strings to numbers and replaced the 9999 to missing.",
        "reason": "Placeholders can distort data if numeric."
    },
    {
        "table": "visits",
        "issue": "weight column named wt_kg in visits_south.csv",
        "decision": "Renamed wt_kg to weight_kg in the south dataframe prior to stacking datasets.",
        "reason": "TO have identical column names for stacking."
    },
    {
        "table": "visits",
        "issue": "sbp sentinel 999 and readings outside 70 through 250",
        "decision": "Coerced to numbers, converted 999 and out-of-range values to missing.",
        "reason": "Removing outliers and avoiding sentinels causing data skew."
    },
    {
        "table": "visits",
        "issue": "impossible and loosely written visit dates",
        "decision": "Enforced strict YYYY-MM-DD format verification.",
        "reason": "Enforcing calendar rules to correct unparsable dates."
    },
    {
        "table": "visits",
        "issue": "repeated patient_id and visit pairs",
        "decision": "Sorted the records chronologically by encounter time and preserved the msot recent bp check.",
        "reason": "Latest checks more reliable, need one per paitnet-visit."
    },
    {
        "table": "sites",
        "issue": "repeated site_code rows",
        "decision": "Filtered and renamed to current names only.",
        "reason": "SIngle site names to prevent join errors."
    }
]

decision_table = pd.DataFrame(decisions_data)
decision_table.to_csv(DECISION_LOG_PATH, index=False)
decision_table


,table,issue,decision,reason
0,patients,exact duplicate rows,Removed exact duplicate rows using drop_duplic...,Identical rows are not possible.
1,patients,gender spellings that are not a code,"Stripped, lowercased, mapped known non-standar...",Correcting categorial gender information.
2,patients,site codes needing normalization,Converted strings to uppercase and stripped ou...,Text fields typed by hand require rigorous for...
3,patients,birth_year sentinel 9999,Coerced strings to numbers and replaced the 99...,Placeholders can distort data if numeric.
4,visits,weight column named wt_kg in visits_south.csv,Renamed wt_kg to weight_kg in the south datafr...,TO have identical column names for stacking.
5,visits,sbp sentinel 999 and readings outside 70 throu...,"Coerced to numbers, converted 999 and out-of-r...",Removing outliers and avoiding sentinels causi...
6,visits,impossible and loosely written visit dates,Enforced strict YYYY-MM-DD format verification.,Enforcing calendar rules to correct unparsable...
7,visits,repeated patient_id and visit pairs,Sorted the records chronologically by encounte...,"Latest checks more reliable, need one per pait..."
8,sites,repeated site_code rows,Filtered and renamed to current names only.,SIngle site names to prevent join errors.


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [ ]:
a1c_raw = raw["a1c_wide"].copy(deep=True)
a1c_long = a1c_raw.melt(
    id_vars=["patient_id"],
    value_vars=["a1c_baseline", "a1c_week12"],
    var_name="visit",
    value_name="a1c_pct"
)
a1c_visit_map = {"a1c_baseline": "baseline", "a1c_week12": "week12"}
a1c_long["visit"] = a1c_long["visit"].map(a1c_visit_map)
assert a1c_long["a1c_pct"].dtype == "float64", "Error: a1c_pct is not float64"
a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)
baseline_results = (a1c_long["visit"] == "baseline").sum()
week12_results = (a1c_long["visit"] == "week12").sum()
baseline_mean = round(a1c_long.loc[a1c_long["visit"] == "baseline", "a1c_pct"].mean(), 2)
week12_mean = round(a1c_long.loc[a1c_long["visit"] == "week12", "a1c_pct"].mean(), 2)
a1c_summary_data = [
    {"visit": "baseline", "results": baseline_results, "a1c_pct_mean": baseline_mean},
    {"visit": "week12", "results": week12_results, "a1c_pct_mean": week12_mean}
]
a1c_summary = pd.DataFrame(a1c_summary_data, columns=["visit", "results", "a1c_pct_mean"])
a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)
a1c_summary

,visit,results,a1c_pct_mean
0,baseline,29107,8.15
1,week12,27653,7.63


### 4.2 Check the keys before joining

In [ ]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]
sites_current = None

In [ ]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

### 4.3 Pivot blood pressure to one row per patient

In [ ]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = None

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [ ]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

### 4.4 Join the analysis table

In [ ]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = None

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [ ]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

### 5.2 Save the analysis extract

In [ ]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

### 5.3 Read the file back

In [ ]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.